# Método por interpolación cuadrática para $\alpha$

Para este método comenzamos seleccionando tres valores $\alpha_1 < \alpha_2 < \alpha_3$ esperando que estén cerca de donde se produce el valor mínimo de $\varphi(\alpha)$ definida anteriormente. Luego construimos el polinomio cuadrático $P(\alpha)$ que se interpola $\varphi(\alpha)$ en $\alpha_1, \alpha_2, \alpha_3$. El mínimo de un polinomio cuadrático se encuentra fácilmente. Definimos $\hat{\alpha}$ en $[\alpha_1,\alpha_3]$ para que $P(\hat{\alpha})$ sea un mínimo en $[\alpha_1,\alpha_3]$ y utilizamos $P(\hat{\alpha})$ para aproximar el valor mínimo de $\varphi(\alpha)$. Entonces  $\hat{\alpha}$ se usa para determinar la nueva iteración para aproximar el valor mínimo de $J$.

Para minimizar el cálculo, primero elegimos $\alpha_1 = 0$. Luego se encuentra $\alpha_3$ con $\varphi(\alpha_3) < \varphi(\alpha_1)$. (Dado que $\alpha_1$ no minimiza $\varphi$, dicho número $\alpha_3$ existe). Finalmente, $\alpha_2$ se elige como $\alpha_3/ 2$.

El valor mínimo de $\varphi$ en $[\alpha_1,\alpha_3]$ se produce en el único punto crítico de $\varphi$ o en el punto final derecho $\alpha_3$ porque, por supuesto, $P(\alpha_3) = \varphi(\alpha_3) < \varphi(\alpha_1) = P(\alpha_1)$. Dado que $P(\alpha)$ es un polinomio cuadrático, el punto crítico se puede encontrar resolviendo una ecuación lineal.

A continuación se presenta una implementación en Python de este algoritmo.

**Ejemplo**

Debemos encontrar entonces un mínimo de la función parabólica del ejemplo anterior dada por

$$
J(\theta_0, \theta_1) = 5 \theta_0^2 + \frac{1}{2} \theta_1^2 - \theta_0 + \theta_1.
$$

Para este ejemplo se usa el módulo Sympy de Python para calcular el gradiente de forma simbólica.

In [1]:
# requirements
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
import pandas as pd
import sympy as sym

plt.style.use('ggplot')
%matplotlib ipympl

In [2]:
def steepestDescent(cost, gradient, theta, tol, M):
    
    theta_rec = theta
    k = 0
    while k < M:
        J1 = cost(*theta)
        grad = np.array(gradient(*theta))
        gradnorm = np.linalg.norm(grad)
        
        if gradnorm == 0:
            print("Zero gradient, the procedure completed and might have a minimum.")
            return theta_rec
        
        unitgrad = grad/gradnorm
        alpha1 = 0
        alpha3 = 1
        J3 = cost(*(theta - alpha3*grad))
        
        while J3 >= J1:
            alpha3 = alpha3/2
            J3 = cost(*(theta - alpha3*grad))
            if alpha3 < tol/2:
                print("No likely improvement, the procedure completed and might have a minimum.")
                return theta_rec
        
        alpha2 = alpha3/2
        J2 = cost(*(theta - alpha2*grad))
        
        # Newton’s forward divided-difference
        h1 = (J2 - J1)/alpha2
        h2 = (J3 - J2)/(alpha3 - alpha2)
        h3 = (h2 - h1)/alpha3
        
        alpha0 = 0.5*(alpha2 - h1/h3)
        J0 = cost(*(theta - alpha0*grad))
        
        idx = np.argmin([J0, J3])
        alphas = [alpha0, alpha3]
        alpha  = alphas[idx]
        J = cost(*(theta - alpha*grad))
        
        theta = theta - alpha*grad
        
        theta_rec = np.vstack((theta_rec, theta)) # Store theta_k 
        
        if np.abs(J - J1) < tol:
            print('The procedure was successful.')
            return theta_rec
        k +=1
        
    print('The method failed after ', M, ' iterations')

In [3]:
# Cost function (symbolic)
theta0, theta1 = sym.symbols('theta_0 theta_1')
cost_sym = 5*theta0**2 + 0.5*theta1**2 - theta0 + theta1
cost_sym

5*theta_0**2 - theta_0 + 0.5*theta_1**2 + theta_1

In [4]:
# Gradient (symbolic)
var = [theta0, theta1]
grad_sym = []
for i in range(len(var)):
    derivative = sym.diff(cost_sym, var[i])
    grad_sym.append(derivative)
    
sym.Matrix(grad_sym)

Matrix([
[ 10*theta_0 - 1],
[1.0*theta_1 + 1]])

In [5]:
# lamdify the functions
cost = sym.lambdify(var, cost_sym, 'numpy')
grad = sym.lambdify(var, grad_sym, 'numpy')

In [6]:
theta = np.array([1, 1])
tol = 0.01
M = 100

theta_rec = steepestDescent(cost, grad, theta, tol, M)
J_rec = cost(theta_rec[:, 0], theta_rec[:, 1])

data = np.hstack((theta_rec, J_rec.reshape(len(J_rec), 1)))
pd.DataFrame(data, columns=["theta0", "theta1", "J(theta)"])

The procedure was successful.


,theta0,theta1,J(theta)
0,1.000000,1.000000,5.500000
1,0.060197,0.791155,1.062039
2,0.339808,-0.467094,-0.120468
3,0.089394,-0.522742,-0.435550
4,0.163897,-0.858006,-0.519504
5,0.097174,-0.872833,-0.541874
6,0.117026,-0.962165,-0.547835


In [7]:
theta0 = np.linspace(-1, 1, 100)
theta1 = np.linspace(-3, 1, 100)
[T0, T1] = np.meshgrid(theta0, theta1)

J = cost(T0, T1)
J_rec = cost(theta_rec[:, 0], theta_rec[:, 1])

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1,1,1, projection="3d")

ax.plot_surface(T0, T1, J, cmap ="jet", alpha=0.5)
ax.contour(T0, T1, J, zdir='z', cmap ="jet", offset = -5)

ax.plot(theta_rec[:, 0], theta_rec[:, 1], J_rec, "r")
ax.plot(theta_rec[:, 0], theta_rec[:, 1], -5, "r")

ax.set_zlim3d(-5, 10)
ax.set_xlabel(r"$\theta_0$")
ax.set_ylabel(r"$\theta_1$")
ax.set_zlabel(r"$J(\theta_0, \theta_1)$")
plt.tight_layout()
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Podemos observar que este método converge más rápido que los métodos explicados anteriormente. Ademas puede ser aplicado a funciones de coste genéricas como veremos a continuación.

**Ejemplo**

Encuentre una aproximación al mínimo de

$$
J(\theta_0, \theta_1) = \cos(\theta_0 + \theta_1) + \sin{\theta_0} + \cos{\theta_1}
$$

In [8]:
# Cost function (symbolic)
theta0, theta1 = sym.symbols('theta_0 theta_1')
cost_sym = sym.cos(theta0 + theta1) + sym.sin(theta0) + sym.cos(theta1)
cost_sym

sin(theta_0) + cos(theta_1) + cos(theta_0 + theta_1)

In [9]:
# Gradient (symbolic)
var = [theta0, theta1]
grad_sym = []
for i in range(len(var)):
    derivative = sym.diff(cost_sym, var[i])
    grad_sym.append(derivative)
    
sym.Matrix(grad_sym)

Matrix([
[-sin(theta_0 + theta_1) + cos(theta_0)],
[-sin(theta_1) - sin(theta_0 + theta_1)]])

In [10]:
# lamdify the functions
cost = sym.lambdify(var, cost_sym, 'numpy')
grad = sym.lambdify(var, grad_sym, 'numpy')

In [11]:
theta = np.array([1, -1])
tol = 0.01
M = 100

theta_rec = steepestDescent(cost, grad, theta, tol, M)
J_rec = cost(theta_rec[:, 0], theta_rec[:, 1])

data = np.hstack((theta_rec, J_rec.reshape(len(J_rec), 1)))
pd.DataFrame(data, columns=["theta0", "theta1", "J(theta)"])

The procedure was successful.


,theta0,theta1,J(theta)
0,1.000000,-1.000000,2.381773
1,0.459698,-1.841471,0.364195
2,-0.684422,-3.026646,-2.467810
3,-0.913644,-2.613631,-2.582113
4,-1.008248,-2.665063,-2.596428
5,-1.034284,-2.617283,-2.597923


In [12]:
theta0 = theta1 = np.linspace(-4, 4, 100)
[T0, T1] = np.meshgrid(theta0, theta1)

J = cost(T0, T1)
J_rec = cost(theta_rec[:, 0], theta_rec[:, 1])

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1,1,1, projection="3d")

ax.plot_surface(T0, T1, J, cmap ="jet", alpha=0.5)
ax.contour(T0, T1, J, zdir='z', cmap ="jet", offset = -5)

ax.plot(theta_rec[:, 0], theta_rec[:, 1], J_rec, "r")
ax.plot(theta_rec[:, 0], theta_rec[:, 1], -5, "r")

ax.set_zlim3d(-5, 2);
ax.set_xlabel(r"$\theta_0$")
ax.set_ylabel(r"$\theta_1$")
ax.set_zlabel(r"$J(\theta_0, \theta_1)$")
plt.tight_layout()
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Otros métodos más sofisticados

- [Conjugate gradient](https://en.wikipedia.org/wiki/Conjugate_gradient_method)
- [Nonlinear conjugate gradient method](https://en.wikipedia.org/wiki/Nonlinear_conjugate_gradient_method)
- [Broyden–Fletcher–Goldfarb–Shanno algorithm](https://en.wikipedia.org/wiki/Broyden%E2%80%93Fletcher%E2%80%93Goldfarb%E2%80%93Shanno_algorithm)
- [Limited-memory BFGS](https://en.wikipedia.org/wiki/Limited-memory_BFGS)
- [Nelder–Mead method](https://en.wikipedia.org/wiki/Nelder%E2%80%93Mead_method)

Algunos de estos métodos se presentan brevemente a continución usando el módulo de Python scypy.optimize

In [13]:
from scipy.optimize import minimize

In [14]:
# Cost function
def cost(theta):
    return 5*theta[0]**2 + 0.5*theta[1]**2 - theta[0] + theta[1]

# Reporter function to capture intermediate states of optimization
def records(theta_k):
    global theta_rec, J_rec
    theta_rec = np.vstack((theta_rec, theta_k))
    J_rec = np.vstack((J_rec, cost(theta_k)))

## Conjugate gradient

In [15]:
theta0 = np.array([1, 1])

theta_rec = [theta0]
J_rec = cost(theta0)

CG_res = minimize(cost, theta0, method='CG', tol=0.01, callback=records)

data = np.hstack((theta_rec, J_rec))
pd.DataFrame(data, columns=["theta0", "theta1", "J(theta)"])

,theta0,theta1,J(theta)
0,1.000000,1.000000,5.500000
1,0.014051,0.780900,1.072739
2,0.017595,-1.190210,-0.497957
3,0.100365,-1.188610,-0.532213
4,0.099849,-0.999939,-0.550000


In [16]:
theta0 = np.linspace(-1, 1, 100)
theta1 = np.linspace(-3, 1, 100)
[T0, T1] = np.meshgrid(theta0, theta1)

J = np.zeros((100, 100))
for i in range(100):
    for j in range(100):
        theta = [theta0[j], theta1[i]]
        J[i, j] = cost(theta)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1,1,1, projection="3d")

ax.plot_surface(T0, T1, J, cmap ="jet", alpha=0.5)
ax.contour(T0, T1, J, zdir='z', cmap ="jet", offset = -5)

ax.plot(theta_rec[:, 0], theta_rec[:, 1], J_rec[:, 0], "r")
ax.plot(theta_rec[:, 0], theta_rec[:, 1], -5, "r")

ax.set_zlim3d(-5, 10)
ax.set_xlabel(r"$\theta_0$")
ax.set_ylabel(r"$\theta_1$")
ax.set_zlabel(r"$J(\theta_0, \theta_1)$")
plt.tight_layout()
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## Broyden–Fletcher–Goldfarb–Shanno algorithm (BFGS)

In [17]:
theta0 = np.array([1, 1])

theta_rec = [theta0]
J_rec = cost(theta0)

CG_res = minimize(cost, theta0, method='BFGS', tol=0.01, callback=records)

data = np.hstack((theta_rec, J_rec))
pd.DataFrame(data, columns=["theta0", "theta1", "J(theta)"])

,theta0,theta1,J(theta)
0,1.000000,1.000000,5.500000
1,0.014051,0.780900,1.072739
2,0.101760,-1.079216,-0.546847
3,0.100001,-1.000000,-0.550000


In [18]:
theta0 = np.linspace(-1, 1, 100)
theta1 = np.linspace(-3, 1, 100)
[T0, T1] = np.meshgrid(theta0, theta1)

J = np.zeros((100, 100))
for i in range(100):
    for j in range(100):
        theta = [theta0[j], theta1[i]]
        J[i, j] = cost(theta)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1,1,1, projection="3d")

ax.plot_surface(T0, T1, J, cmap ="jet", alpha=0.5)
ax.contour(T0, T1, J, zdir='z', cmap ="jet", offset = -5)

ax.plot(theta_rec[:, 0], theta_rec[:, 1], J_rec[:, 0], "r")
ax.plot(theta_rec[:, 0], theta_rec[:, 1], -5, "r")

ax.set_zlim3d(-5, 10)
ax.set_xlabel(r"$\theta_0$")
ax.set_ylabel(r"$\theta_1$")
ax.set_zlabel(r"$J(\theta_0, \theta_1)$")
plt.tight_layout()
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …